### Импорты и загрузка датасета

In [12]:
import torch
import torch.nn as nn
import random
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import resnet18 # для сравнение с текущей моделью resnet20

# Нужно для воспроизводимости результатов и для чистоты экспериментов (разность результатов будет не из-за различных чисел рандомайзера). 
# На разных машинах и библиотеках могут отличаться данные.
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True

# Загрузка датасета
import torchvision.datasets
CIFAR_train = torchvision.datasets.CIFAR10('./', download=True, train=True)
CIFAR_test = torchvision.datasets.CIFAR10('./', download=True, train=False)

Files already downloaded and verified
Files already downloaded and verified


### Инициализация тренировочных и тестовых тензеров с данными и их настройка.

In [13]:
X_train = torch.FloatTensor(CIFAR_train.data)
y_train = torch.LongTensor(CIFAR_train.targets)
X_test = torch.FloatTensor(CIFAR_test.data)
y_test = torch.LongTensor(CIFAR_test.targets)

# Переводим данные пикселей в диапазон от 0 до 1.
X_train /= 255.
X_test /= 255.

# Устанавливаем порядок размерностей тензора. В порядке: Batches, colors, height, widht.
X_train = X_train.permute(0, 3, 1, 2)
X_test = X_test.permute(0, 3, 1, 2)

### Реализация класса **ResNet20**

In [14]:
class BasicBlock(torch.nn.Module):
    """
    Базовый блок для ResNet (например, ResNet-18, ResNet-34).
    Состоит из двух сверток 3x3, Batch Normalization, ReLU и пропускающего соединения.
    """
    # Масштабирующий фактор для выходных каналов (для более глубоких ResNet)
    # В базовом блоке он равен 1, так как количество каналов не меняется внутри блока
    # (кроме случая понижения размерности через stride)
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        """
        Инициализация блока.

        Аргументы:
            in_channels (int): Количество входных каналов (из предыдущего слоя/блока).
            out_channels (int): Количество выходных каналов, генерируемых этим блоком.
                                 Свертки внутри блока будут иметь это количество фильтров.
            stride (int): Шаг (stride) для первой свертки. Если stride=2,
                          происходит понижение пространственной размерности (downsampling).
                          По умолчанию 1 (размерность сохраняется).
            downsample (nn.Module, optional): Модуль для преобразования пропускающего
                                              соединения, если нужно изменить размерность
                                              (из-за stride=2 или разного числа каналов).
                                              Обычно это свертка 1x1 + BatchNorm.
                                              По умолчанию None.
        """
        super(BasicBlock, self).__init__()

        # Сохраняем шаг
        self.stride = stride

        # Первая свертка: 3x3, с шагом 'stride', padding=1 для сохранения размера при stride=1
        # bias=False, так как Batch Normalization имеет свои параметры смещения и масштабирования
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        # Пакетная нормализация после первой свертки
        self.bn1 = torch.nn.BatchNorm2d(out_channels)  
        
        # Вторая свертка: 3x3, шаг всегда 1
        self.conv2 = torch.nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = torch.nn.BatchNorm2d(out_channels) 

        # Функция активации ReLU (inplace=True для экономии памяти)
        self.relu = torch.nn.ReLU(inplace=True) 

        # Сохраняем слой для преобразования пропускающего соединения (если он нужен)
        self.downsample = downsample
        

    def forward(self, x):
         # Сохраняем входное значение для пропускающего соединения
        identity = x

         # --- Основной путь ---
        # Слой 1: Conv -> BN -> ReLU
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # Слой 2: Conv -> BN
        out = self.conv2(out)
        out = self.bn2(out)
        # --------------------

        # --- Пропускающее соединение (Skip Connection) ---
        # Если необходимо преобразовать вход 'identity' (из-за stride или числа каналов),
        # используем слой 'downsample'
        if self.downsample is not None:
            identity = self.downsample(x)
        # -----------------------------------------------

        # --- Сложение выхода основного пути и пропускающего соединения ---
        # Это и есть ключевая идея ResNet: H(x) = F(x) + x
        # где F(x) - выход основного пути (out), x - вход (identity)
        out += identity
         # ------------------------------------------------------------

        # --- Финальная активация ReLU ---
        # Применяется *после* сложения
        out = self.relu(out)
        # -----------------------------

        return out




class ResNet20(nn.Module):
    def __init__(self, block=BasicBlock, num_blocks_list=[3, 3, 3], num_classes=10, base_channels=16):
        """
        Инициализация ResNet-20 для CIFAR-10 (вход 3x32x32).

        Args:
            block (nn.Module): Тип блока (BasicBlock).
            num_blocks_list (list): Список с количеством блоков в каждой стадии. [3, 3, 3] для ResNet-20.
            num_classes (int): Количество выходных классов.
            base_channels (int): Начальное количество каналов.
        """
        super(ResNet20, self).__init__()
        self.in_channels = base_channels # Текущее количество каналов для следующего слоя

        # Начальный слой
        self.conv1 = nn.Conv2d(3, self.in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)

        # Стадии ResNet
        self.layer1 = self._make_layer(block, base_channels, num_blocks_list[0], stride=1)
        self.layer2 = self._make_layer(block, base_channels*2, num_blocks_list[1], stride=2)
        self.layer3 = self._make_layer(block, base_channels*4, num_blocks_list[2], stride=2)

        # Финальный слой
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) # Global Average Pooling
        self.fc = nn.Linear(base_channels*4 * block.expansion, num_classes) # block.expansion = 1 для BasicBlock

    def _make_layer(self, block, out_channels, num_blocks, stride):
        """
        Вспомогательная функция для создания стадии ResNet (последовательности блоков).
        """
        downsample = None
        # Условие для downsample: шаг не равен 1 ИЛИ количество входных/выходных каналов не совпадает
        if stride != 1 or self.in_channels != out_channels * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * block.expansion),
            )

        layers = []
        # Первый блок стадии может иметь stride > 1 и downsample
        layers.append(block(self.in_channels, out_channels, stride, downsample))

        # Обновляем in_channels для следующих блоков этой стадии
        self.in_channels = out_channels * block.expansion

        # Добавляем остальные блоки стадии (stride=1, без downsample внутри блока)
        for _ in range(1, num_blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x) # Не забываем ReLU

        x = self.layer1(x) # Стадия 1 (16 каналов, 32x32)
        x = self.layer2(x) # Стадия 2 (32 канала, 16x16)
        x = self.layer3(x) # Стадия 3 (64 канала, 8x8)

        x = self.avgpool(x) # Global Average Pooling -> (batch, 64, 1, 1)
        x = torch.flatten(x, 1) # Flatten -> (batch, 64)
        x = self.fc(x) # Linear -> (batch, 10)

        return x

In [15]:
def train(net, X_train, y_train, X_test, y_test):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    net = net.to(device)
    loss = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1.0e-3)
    
    batch_size = 256

    test_accuracy_history = []
    test_loss_history = []

    X_test = X_test.to(device)
    y_test = y_test.to(device)

    for epoch in range(20):
        order = np.random.permutation(len(X_train))
        for start_index in range(0, len(X_train), batch_size):
            optimizer.zero_grad()
            net.train()

            batch_indexes = order[start_index:start_index+batch_size]

            X_batch = X_train[batch_indexes].to(device)
            y_batch = y_train[batch_indexes].to(device)

            preds = net.forward(X_batch)

            loss_value = loss(preds, y_batch)
            loss_value.backward()

            optimizer.step()
            
            X_batch

        net.eval()
        test_preds = net.forward(X_test)
        test_loss_history.append(loss(test_preds, y_test).data.cpu())

        accuracy = (test_preds.argmax(dim=1) == y_test).float().mean().data.cpu()
        test_accuracy_history.append(accuracy)

        print(accuracy)
    del net
    return test_accuracy_history, test_loss_history

accuracies = {}
losses = {}


In [ ]:
accuracies['ResNet18'], losses['ResNet18'] = \
    train(ResNet20(), X_train, y_train, X_test, y_test)

accuracies['ResNet18'], losses['ResNet18'] = \
    train(resnet18(), X_train, y_train, X_test, y_test)

KeyboardInterrupt: 